In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# Joints on the wire: the SO-101 server node and arm client

The camera pattern, applied to the thing that can actually pinch you.
**Exactly one process owns the Feetech serial bus**: the server node.
Everything else speaks `sensor_msgs/JointState` over two topics.

![Alt Text](../../animations/SO-101_Loop_Animation.gif)

**This notebook moves the real arm.** The server below connects straight
to the SO-101's serial bus, so every command in this notebook turns into
physical motion. Clear the space around the arm and keep hands away from
the joints.

The conventions that make this safe (and your TODOs implement):

- joints are **degrees**, keyed `shoulder_pan, shoulder_lift, elbow_flex,
  wrist_flex, wrist_roll, gripper`; **partial commands are legal**;
- every command passes `clamp_joints`, on the client *and* the server
  (belt and braces);
- the state topic's `effort` carries the **gripper servo current**, with
  **NaN meaning "not readable"**. That feedback lets the grip behavior
  stop squeezing at first contact instead of crushing whatever it holds.

**Matching TODOs:** `2.4`-`2.9` in `project/ros/arm_node.py` and
`project/ros/arm_client.py`.

In [ ]:
# workshop bootstrap: run against the finished reference code in solution/
# (no TODO stubs), so every cell works out of the box. To exercise YOUR
# implementations instead, change "solution" to "project" and restart.
import sys, pathlib
import threading, time
import math

PROJECT = (pathlib.Path.cwd() / ".." / ".." / "solution").resolve()
assert (
    PROJECT / "config" / "workshop.yaml"
).exists(), f"solution not found at {PROJECT}"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from common.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

from common.config import load_config, resolve, PROJECT_ROOT
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events
from common.motion import MOTOR_NAMES, REST_POSE, clamp_joints, make_arm_backend
from common.ros_helpers import joint_state_msg
from common.motion import ArmClient
from common.ros_helpers import parse_joint_state
from common.motion import JOINT_LIMITS

cfg = load_config()
print("running against:", PROJECT_ROOT)

In [ ]:
# ROS 2 check: this notebook runs on real ROS 2 (rclpy) only.
try:
    import rclpy

    HAVE_ROS = True
except ImportError:
    HAVE_ROS = False

ROS_HELP = """rclpy is not importable in this kernel, so the ROS 2 cells cannot run."""


def require_ros():
    if not HAVE_ROS:
        raise RuntimeError(ROS_HELP)


if HAVE_ROS:
    if not rclpy.ok():
        rclpy.init()
    print("real ROS 2 (rclpy) ready")
else:
    print(ROS_HELP)

> **About background nodes.** The demo nodes this notebook creates are
> self-contained: they publish their own data, so the lesson works with
> nothing else running. To also run the *real* device nodes (the SO-101
> arm, the cameras, the intent node), start them yourself in a terminal;
> the "On the rig" section below has the exact commands. Every subscriber
> simply reads whatever is on the topic, so both can coexist.

In [ ]:
require_ros()

# A tiny helper to spin nodes on background executors, so the notebook
# stays interactive while nodes tick. (This mirrors how the pipeline's
# orchestrator embeds its client nodes.)


class Runner:
    def __init__(self, node):
        from rclpy.executors import SingleThreadedExecutor

        self.node, self.executor = node, SingleThreadedExecutor()
        self.executor.add_node(node)
        threading.Thread(target=self.executor.spin, daemon=True).start()

    def stop(self):
        self.executor.shutdown()
        try:
            self.node.destroy_node()
        except Exception:
            pass


def wait_for(pred, timeout_s=3.0, interval=0.02):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        v = pred()
        if v:
            return v
        time.sleep(interval)
    return None

## 1. A minimal arm server

The demo twin of TODOs 2.4-2.6, with one difference from the camera
notebook: the backend is the real thing. `make_arm_backend(cfg)` connects
a `LeRobotArm` to the Feetech serial bus using the `robot:` section of the
workshop config, and this notebook becomes the ONE process that owns the
bus. If the arm isn't reachable, the cell tells you what to check instead
of running.

Note the **control thread**: subscriptions land on executor threads, but
only one sequential loop touches the arm. That loop is the only thing on
the serial bus. No contention, ever.

In [ ]:
require_ros()

from rclpy.node import Node
from rclpy.qos import HistoryPolicy, QoSProfile, ReliabilityPolicy
from sensor_msgs.msg import JointState

SENSOR_QOS = QoSProfile(
    reliability=ReliabilityPolicy.BEST_EFFORT, history=HistoryPolicy.KEEP_LAST, depth=1
)


def open_real_arm():
    """Connect to the real SO-101 over the Feetech serial bus."""
    try:
        return make_arm_backend(cfg)  # LeRobotArm per the robot: config
    except Exception as e:
        raise RuntimeError(
            "could not connect to the SO-101. To fix:\n"
            f"  - check the USB cable and that {cfg['robot']['motor_port']} exists\n"
            "  - check port permissions (sudo usermod -aG dialout $USER, then log out/in)\n"
            "  - make sure nothing else owns the bus (a running arm_node, another kernel)\n"
            "  - if this arm is new to this machine, calibrate once in a terminal first\n"
            f"then re-run this cell. (underlying error: {e})"
        ) from e


class DemoArmServer(Node):
    def __init__(self, control_fps=60.0):
        super().__init__("demo_so101_server")
        self.arm = open_real_arm()
        self._lock = threading.Lock()
        self._target = dict(REST_POSE)
        self._fresh_until = 0.0
        self._last_sent = None
        self.create_subscription(
            JointState, "/so101/joint_command", self.on_command, SENSOR_QOS
        )
        self.state_pub = self.create_publisher(
            JointState, "/so101/joint_state", SENSOR_QOS
        )
        self._running = True
        threading.Thread(
            target=self.control_loop, args=(control_fps,), daemon=True
        ).start()

    def on_command(self, msg):
        joints = {
            n: float(p) for n, p in zip(msg.name, msg.position) if n in MOTOR_NAMES
        }  # partial dicts OK
        if joints:
            with self._lock:
                self._target.update(clamp_joints(joints))  # safety backstop
                self._fresh_until = time.time() + 1.0

    def control_loop(self, fps):
        # The workshop config drives the arm the way the pipeline (and
        # gesture_mimic) does: `max_relative_target: null`, so LeRobot's
        # per-tick slew clamp is OFF, a command is ONE absolute goal write,
        # and the servo's onboard profile produces the motion. That means:
        # only write a goal when it CHANGES - re-writing the same goal every
        # tick restarts the profile and the motion turns twitchy.
        while self._running:
            with self._lock:
                target, fresh = dict(self._target), time.time() < self._fresh_until
            if fresh and target != self._last_sent:
                self.arm.send_joints(target)  # absolute goal, one write
                self._last_sent = target
            joints = self.arm.get_joints()
            load = self.arm.get_gripper_load()  # None if not readable
            self.state_pub.publish(
                joint_state_msg(
                    MOTOR_NAMES,
                    [joints[m] for m in MOTOR_NAMES],
                    effort=[
                        math.nan if (load is None or m != "gripper") else float(load)
                        for m in MOTOR_NAMES
                    ],
                    node=self,
                )
            )
            time.sleep(1.0 / fps)


server = DemoArmServer()
server_runner = Runner(server)
print("arm server up - sole owner of the real Feetech bus")

## 2. The client side, and the first rule of robot clients

**Never command an arm whose state you can't see.** The client blocks until
the state stream is live, *then* exposes the `ArmClient` API. (TODO 2.7's
provided wait-loop does the same.)

In [ ]:
require_ros()


class DemoArmClient(ArmClient):
    # The demo implements the two primitives (send/get); everything composed
    # (go_to, move_to, go_to_rest) comes from the shared ArmClient base.
    # _direct_goal matches the server's mode (max_relative_target: null):
    # goals are sent ONCE and the servo's onboard profile makes the motion.
    _direct_goal = True

    def __init__(self, timeout_s=4.0):
        from rclpy.executors import SingleThreadedExecutor

        self._lock, self._state, self._effort = threading.Lock(), None, None
        self.node = rclpy.create_node("demo_arm_client")
        self.pub = self.node.create_publisher(
            JointState, "/so101/joint_command", SENSOR_QOS
        )
        self.node.create_subscription(
            JointState, "/so101/joint_state", self.on_state, SENSOR_QOS
        )
        self._exec = SingleThreadedExecutor()
        self._exec.add_node(self.node)
        threading.Thread(target=self._exec.spin, daemon=True).start()
        assert wait_for(
            lambda: self._state is not None, timeout_s
        ), "no state stream - is the server node running?"

    def on_state(self, msg):
        positions, effort = parse_joint_state(msg)  # NaN->None handled here
        with self._lock:
            self._state, self._effort = positions, effort

    def send_joints(self, joints):
        joints = clamp_joints(joints)  # client-side clamp too
        self.pub.publish(
            joint_state_msg(list(joints), list(joints.values()), node=self.node)
        )

    def get_joints(self):
        with self._lock:
            return dict(self._state)

    def get_gripper_load(self):
        with self._lock:
            return self._effort


arm = DemoArmClient()
print(
    "state stream live. rest pose:",
    {k: round(v, 1) for k, v in arm.get_joints().items()},
)
print(
    "gripper effort:",
    arm.get_gripper_load(),
    " (the servo current; None means this firmware can't report it, NOT zero)",
)

## 3. Drive every joint by hand

Before scripting motion, get a feel for the real arm. One slider per
joint, ranges straight from `JOINT_LIMITS`. Drag a slider and **release**
it: the release value is published as one absolute goal, the server merges
it into its target pose, and the servo's onboard profile produces the
motion. One gesture, one write - exactly how the pipeline commands
point-to-point moves in direct-goal mode (`max_relative_target: null`).
(Why on release? Re-writing goals every tick restarts the servo profile -
section 1's lesson - and a running cell only receives a drag's flood of
widget syncs throttled, while one sync per gesture always lands.)

The sliders start at `REST_POSE` (wrist_flex rests at 22.5 on this rig)
and the cell sends the arm there first, so the UI and the metal agree from
the first pixel. The gripper slider spans the jaws' calibrated range,
43 (closed) to 95 (open).

Two extras:

- **link shoulder_lift + elbow_flex** (on by default): the two proximal
  joints move together, in opposition - move shoulder_lift by +0.1 and
  elbow_flex follows by -0.1, in the SAME command - which keeps the
  forearm attitude roughly steady as the shoulder moves. Untick to drive
  them independently.
- **rest pose** sends ONE full-pose command (six separate partial
  commands in a burst would race the depth-1 best-effort queue and drop),
  then quietly syncs the sliders to match.

The loop below just keeps UI events flowing and streams the *measured*
joint state back under the sliders. Press **stop** to end the loop and
move to the next cell.

These sliders move the physical arm. Move them gently and keep clear.


In [ ]:
require_ros()

LINKED = ("shoulder_lift", "elbow_flex")  # move together, in opposition

arm.send_joints(dict(REST_POSE))  # start from a known pose: rest -
# the sliders below default to it too
sliders = {}
for name in MOTOR_NAMES:
    lo, hi = JOINT_LIMITS[name]
    sliders[name] = widgets.FloatSlider(
        value=REST_POSE[name],
        min=lo,
        max=hi,
        step=0.1 if name in LINKED else 0.5,
        description=name,
        continuous_update=False,  # one state sync per gesture, on release
        readout_format=".1f",
        layout=widgets.Layout(width="440px"),
        style={"description_width": "110px"},
    )

link_chk = widgets.Checkbox(
    value=True,
    indent=False,
    description="link shoulder_lift + elbow_flex (opposite)",
    layout=widgets.Layout(width="340px"),
)

# Event-driven commands: each slider CHANGE publishes one partial command;
# the server merges it into the full target and writes the bus once. With
# the link on, a shoulder_lift move of +d takes elbow_flex along by -d
# (and vice versa) in the SAME command, so the pair stays coordinated.
syncing = {"on": False}  # guards programmatic slider writes


def make_sender(name):
    def _send(change):
        if syncing["on"]:
            return
        cmd = {name: float(change["new"])}
        if link_chk.value and name in LINKED:
            other = LINKED[1] if name == LINKED[0] else LINKED[0]
            delta = float(change["new"]) - float(change["old"])
            partner = sliders[other]
            target = max(partner.min, min(partner.max, partner.value - delta))
            syncing["on"] = True
            try:
                partner.value = target  # mirror the UI without re-sending
            finally:
                syncing["on"] = False
            cmd[other] = float(target)
        arm.send_joints(cmd)  # ONE message for the whole gesture

    return _send


senders = {name: make_sender(name) for name in sliders}
for name, s in sliders.items():
    s.observe(senders[name], "value")

stop_btn = widgets.Button(description="stop", button_style="danger")
rest_btn = widgets.Button(description="rest pose", button_style="warning")
stopped = {"flag": False}
stop_btn.on_click(lambda _: stopped.update(flag=True))


def go_rest(_):
    arm.send_joints(dict(REST_POSE))  # ONE full-pose command...
    syncing["on"] = True  # ...then sync the UI quietly
    try:
        for n in MOTOR_NAMES:
            sliders[n].value = REST_POSE[n]
    finally:
        syncing["on"] = False


rest_btn.on_click(go_rest)

readout = widgets.Label()
display(
    widgets.VBox(
        [*sliders.values(), link_chk, widgets.HBox([rest_btn, stop_btn]), readout]
    )
)

# The loop's only jobs: deliver the widget events (poll) and stream the
# MEASURED state back so you can watch the servo track each goal.
with ui_events() as poll:
    while not stopped["flag"]:
        poll(10)  # slider releases + clicks land here
        j = arm.get_joints()
        readout.value = "state:  " + "   ".join(f"{n}={j[n]:.1f}" for n in MOTOR_NAMES)
        time.sleep(0.03)

for name, s in sliders.items():  # detach: the sliders go quiet
    s.unobserve(senders[name], "value")  # once you leave this cell
print("joint control loop stopped - sliders detached")

## 4. The safety clamp survives the transport

`clamp_joints` runs on both ends. Command something insane and watch physics
stay bounded. Heads up: the arm really will swing to its +86 degree pan
limit and back (the software clamp - the mechanical range pulled in 25% so
the arm clears the environment), so keep the space around it open anyway.

In [ ]:
require_ros()

pan_lo, pan_hi = JOINT_LIMITS["shoulder_pan"]
arm.send_joints({"shoulder_pan": 500.0})  # 500 degrees into an +/-86 degree clamp
time.sleep(0.3)
pan = arm.get_joints()["shoulder_pan"]
print(f"commanded 500 -> state reports {pan:.1f}  (limit: {pan_lo:+.0f}/{pan_hi:+.0f})")
arm.send_joints({"shoulder_pan": 0.0})
time.sleep(0.2)

## 5. Why the effort field exists

The gripper servo reports its **Present_Current** through the bus, and over
ROS 2 that current rides in `JointState.effort`, sampled every few control
ticks. Watching current instead of commanded position is what lets a grip
behavior stop closing at first sustained contact rather than driving to a
target and crushing the object on the way.

Two conventions make it work over any transport, and you implement both:
NaN on the wire means "not readable" and the client maps it to `None`,
never zero; when effort is unavailable, behaviors fall back to detecting a
position stall instead.

## 6. On the rig

```bash
# Terminal 1 - the server owns the bus (or --dry-run without hardware):
python -m ros.arm_node --dry-run

# Terminal 2 - watch it like an operator:
ros2 topic echo /so101/joint_state     # positions + effort[gripper]

# Terminal 3 - your client's round-trip test:
python -m ros.arm_client
```

Real-hardware notes (from the pipeline README): calibrate once in a
foreground terminal before backgrounding the server; check `motor_port`
permissions; the server parks the arm at `REST_POSE` on any shutdown path.

## 8. Your turn

Open **`project/ros/arm_node.py`** and **`project/ros/arm_client.py`**:

| TODO | What | Taught in |
|---|---|---|
| **2.4** | server `_setup_ros`: cmd sub + state pub | section 1 |
| **2.5** | server `_on_command`: filter, clamp, store | sections 1 and 4 |
| **2.6** | server `_publish_state`: positions + effort/NaN | sections 1 and 5 |
| **2.7** | client `_setup_ros`: pub + sub + executor | section 2 |
| **2.8** | client `send_joints`: clamp + publish | sections 2 and 4 |
| **2.9** | client `_on_state`: positions + effort->None | sections 2 and 5 |


Then, from the `project/` directory:


```bash
python -m ros.selftest

# to see all the TODOs you have left
python app.py
```



Next: `03_intent_node.ipynb`, language becomes a node.